# LAB03 · Ensambles, reducción dimensional y frontera de Pareto (Green AI)

**Protocolo congelado del Ejercicio 01.** Mismo dataset (**Adult / Census Income**, UCI, CC BY 4.0),
mismo target (`income`: `<=50K` / `>50K`) y **misma partición estratificada 80/20 con semilla 42**.
Nada de lo que se hace aquí cambia el preprocesamiento aprobado: imputación por mediana/moda,
estandarización de las numéricas y codificación one-hot de las categóricas, todo dentro del pipeline.

**Pregunta.** Entre seis o más configuraciones comparables de forma homogénea (lineal, SVM,
Random Forest, boosting y una variante con reducción por PCA), ¿cuál ofrece la mejor relación entre
utilidad predictiva y costo computacional, y qué se pierde al reducir la dimensionalidad?

**Qué se llama Green AI en este laboratorio.** El tiempo de ajuste, el tiempo de inferencia y el
tamaño del modelo serializado son ** proxies contextuales de costo**: permiten comparar configuraciones
en el mismo hardware. **No** son una medición de consumo energético, y el hardware queda registrado
para que las cifras solo valgan en ese contexto.

In [1]:
import json
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import accuracy_score, f1_score, recall_score

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from inf8239_u01.data import DATASET_SPECS, load_dataset
from inf8239_u01.evaluation import stratified_split
from inf8239_u01.green import (
    CATALOGUE,
    build_configuration,
    build_preprocessor,
    environment_report,
    fit_and_measure,
    pareto_flags,
)

REPORTS = ROOT / "reports"
MODELS = ROOT / "models"
REPORTS.mkdir(exist_ok=True)
MODELS.mkdir(exist_ok=True)

entorno = environment_report()
for clave, valor in entorno.items():
    print(f"{clave:15s}: {valor}")

python         : 3.12.15
sistema        : Linux-6.12.113+deb13-amd64-x86_64-with-glibc2.41
procesador     : x86_64
scikit_learn   : 1.9.1
numpy          : 2.5.4
pandas         : 3.1.0


## 1. Dataset, target y partición idénticos al Ejercicio 01

La partición se vuelve a construir con la misma función y la misma semilla. Se muestra una huella
de los índices para comprobar que es exactamente la misma que se usó en LAB02.

In [2]:
DATASET = "adult"
spec = DATASET_SPECS[DATASET]
frame = load_dataset(DATASET, ROOT / "data" / "raw" / f"{DATASET}.csv")

features = frame.drop(columns=spec["exclude"] + [spec["target"]])
target = frame[spec["target"]].astype(str)

X_train, X_test, y_train, y_test = stratified_split(features, target)
print("Entrenamiento:", X_train.shape, "| Prueba:", X_test.shape)
print("Balance prueba:", y_test.value_counts().to_dict())

import hashlib
huella = hashlib.sha256(pd.util.hash_pandas_object(X_test, index=True).values.tobytes()).hexdigest()[:16]
print("Huella de la partición de prueba:", huella, "(debe coincidir con el Ejercicio 01)")

Entrenamiento: (39073, 10) | Prueba: (9769, 10)
Balance prueba: {'<=50K': 7431, '>50K': 2338}
Huella de la partición de prueba: bcc65794964280f8 (debe coincidir con el Ejercicio 01)


## 2. Catálogo homogéneo de configuraciones

Seis familias sobre el mismo preprocesamiento y la misma partición: regresión logística, SVM con dos
valores de `C`, Random Forest con 100 y 300 árboles, boosting (`HistGradientBoosting`) y dos variantes
con reducción por **PCA** (20 componentes). El preprocesamiento produce una matriz **densa** porque el
boosting y el PCA no admiten entradas dispersas: ese es el costo en memoria que la reducción por PCA
pretende compensar.

Cada configuración se ajusta **tres veces** y se reporta la **mediana** del tiempo; la inferencia se
mide tres veces sobre el mismo conjunto de prueba y también se reporta la mediana.

In [3]:
filas = []
artefactos = {}
for nombre in CATALOGUE:
    resultado = fit_and_measure(nombre, X_train, y_train, X_test, repetitions=3)
    pipeline = resultado.pop("pipeline")
    predicciones = resultado.pop("predictions")
    artefactos[nombre] = pipeline
    filas.append({
        **resultado,
        "accuracy": accuracy_score(y_test, predicciones),
        "f1_macro": f1_score(y_test, predicciones, average="macro"),
        "recall_mas_50k": recall_score(y_test, predicciones, pos_label=">50K"),
        "recall_menos_50k": recall_score(y_test, predicciones, pos_label="<=50K"),
    })
    print(f"{nombre:14s} F1 macro={filas[-1]['f1_macro']:.4f} | ajuste={filas[-1]['fit_time_median_s']:7.2f}s "
          f"| inferencia={filas[-1]['predict_time_median_ms']:7.1f}ms | tamaño={filas[-1]['model_size_kb']:9.1f}KB")

resultados = pd.DataFrame(filas).sort_values("f1_macro", ascending=False).reset_index(drop=True)
resultados["is_pareto"] = pareto_flags(resultados)
resultados.to_csv(REPORTS / "green_ai_results.csv", index=False)
resultados

logreg         F1 macro=0.7792 | ajuste=   1.86s | inferencia=   28.1ms | tamaño=      4.2KB


logreg_pca     F1 macro=0.7749 | ajuste=   0.58s | inferencia=   20.1ms | tamaño=     14.1KB


svm_c1         F1 macro=0.7876 | ajuste=  29.57s | inferencia= 5250.8ms | tamaño=   6002.0KB


svm_c10        F1 macro=0.7926 | ajuste=  43.92s | inferencia= 5228.0ms | tamaño=   5979.6KB


svm_c10_pca    F1 macro=0.7888 | ajuste=  31.32s | inferencia= 4304.9ms | tamaño=   2278.8KB


rf_100         F1 macro=0.7823 | ajuste=   0.38s | inferencia=   89.3ms | tamaño=  98807.1KB


rf_300         F1 macro=0.7834 | ajuste=   0.92s | inferencia=  145.4ms | tamaño= 296865.8KB


boost          F1 macro=0.8185 | ajuste=   0.56s | inferencia=   24.4ms | tamaño=    360.4KB


,configuracion,familia,reduce_pca,fit_time_median_s,fit_time_min_s,fit_time_max_s,predict_time_median_ms,model_size_kb,repeticiones,accuracy,f1_macro,recall_mas_50k,recall_menos_50k,is_pareto
0,boost,Boosting,False,0.555829,0.536336,0.645199,24.404811,360.364258,3,0.875320,0.818527,0.659966,0.943076,True
1,svm_c10,SVM,False,43.915197,42.220810,45.998432,5227.984497,5979.615234,3,0.858532,0.792635,0.615911,0.934867,False
2,svm_c10_pca,SVM + PCA,True,31.316008,30.928965,31.392437,4304.930922,2278.824219,3,0.858020,0.788751,0.596236,0.940385,False
3,svm_c1,SVM,False,29.572612,29.130300,30.457188,5250.807410,6002.009766,3,0.857611,0.787623,0.592387,0.941058,False
4,rf_300,Random Forest,False,0.920093,0.895761,0.949581,145.442360,296865.822266,3,0.847989,0.783419,0.630881,0.916297,False
5,rf_100,Random Forest,False,0.379097,0.370101,0.521313,89.324183,98807.110352,3,0.846863,0.782265,0.631309,0.914682,True
6,logreg,lineal,False,1.860137,1.442094,2.205447,28.117361,4.243164,3,0.850650,0.779167,0.588537,0.933118,False
7,logreg_pca,lineal + PCA,True,0.576037,0.436357,0.595991,20.081348,14.147461,3,0.848500,0.774862,0.577844,0.933656,False


In [4]:
# Modelos serializados: cada configuración queda guardada para poder reutilizarla
for nombre, pipeline in artefactos.items():
    ruta = MODELS / f"{nombre}.joblib"
    joblib.dump(pipeline, ruta)
    print(f"{nombre:14s} -> {ruta.name} ({ruta.stat().st_size / 1024:.1f} KB)")

logreg         -> logreg.joblib (6.3 KB)
logreg_pca     -> logreg_pca.joblib (16.4 KB)
svm_c1         -> svm_c1.joblib (6004.3 KB)
svm_c10        -> svm_c10.joblib (5981.9 KB)
svm_c10_pca    -> svm_c10_pca.joblib (2281.2 KB)


rf_100         -> rf_100.joblib (98810.7 KB)


rf_300         -> rf_300.joblib (296872.8 KB)
boost          -> boost.joblib (369.8 KB)


## 3. Frontera de Pareto: utilidad frente a costo

Una configuración está **dominada** si otra ofrece igual o mejor F1 macro con igual o menor tiempo de
ajuste. Solo las no dominadas pueden ser la mejor opción: elegir siempre la de mayor F1 macro ignora
el costo, y elegir siempre la más barata ignora la utilidad.

In [5]:
figura, (eje, eje2) = plt.subplots(1, 2, figsize=(14.5, 6))
colores = {"Boosting": "#0f7c7b", "Random Forest": "#b45309", "SVM": "#2563eb",
           "SVM + PCA": "#dc2626", "lineal": "#7c3aed", "lineal + PCA": "#a16207"}
for familia, grupo in resultados.groupby("familia"):
    eje.scatter(grupo["fit_time_median_s"], grupo["f1_macro"], label=familia, s=80, alpha=0.85,
                color=colores.get(familia))
    eje2.scatter(grupo["model_size_kb"] / 1024, grupo["f1_macro"], label=familia, s=80, alpha=0.85,
                 color=colores.get(familia))
frontera = resultados[resultados["is_pareto"]].sort_values("fit_time_median_s")
eje.plot(frontera["fit_time_median_s"], frente := frontera["f1_macro"], "--", color="#334155", linewidth=1.3,
         label="frontera de Pareto")
for _, fila in resultados.iterrows():
    eje.annotate(fila["configuracion"], (fila["fit_time_median_s"], fila["f1_macro"]),
                 textcoords="offset points", xytext=(6, 5), fontsize=8, color="#475569")
    eje2.annotate(fila["configuracion"], (fila["model_size_kb"] / 1024, fila["f1_macro"]),
                  textcoords="offset points", xytext=(6, 5), fontsize=8, color="#475569")
eje.set_xlabel("Tiempo de ajuste (mediana de 3 repeticiones, s)")
eje.set_ylabel("F1 macro en la prueba")
eje.set_title("Utilidad frente a tiempo de ajuste")
eje.set_xscale("log")
eje.grid(alpha=0.25)
eje.legend(fontsize=8, ncol=2)
eje2.set_xlabel("Tamaño del pipeline serializado (MB, escala logarítmica)")
eje2.set_ylabel("F1 macro en la prueba")
eje2.set_title("Utilidad frente a tamaño del modelo")
eje2.set_xscale("log")
eje2.grid(alpha=0.25)
eje2.legend(fontsize=8, ncol=2)
figura.suptitle("Adult · utilidad frente a costo computacional (Green AI)")
figura.tight_layout()
figura.savefig(REPORTS / "pareto.png", dpi=160, bbox_inches="tight")
plt.close(figura)
print("Guardado:", REPORTS / "pareto.png")
print("\nFrontera de Pareto:")
print(resultados[resultados["is_pareto"]][["configuracion", "f1_macro", "fit_time_median_s",
                                          "predict_time_median_ms", "model_size_kb"]].to_string(index=False))

Guardado: /home/trek/source/uasd/cienciadedatos/tareas/Unidad_1/INF8239_U01/reports/pareto.png

Frontera de Pareto:
configuracion  f1_macro  fit_time_median_s  predict_time_median_ms  model_size_kb
        boost  0.818527           0.555829               24.404811     360.364258
       rf_100  0.782265           0.379097               89.324183   98807.110352


## 4. PCA: cuánta información se conserva

El PCA se ajusta **dentro** del pipeline sobre el entrenamiento y se aplica a la prueba, de modo que
no hay fuga. Se mide cuánta varianza explican los componentes y qué parte sobrevive en dos
dimensiones.

In [6]:
preprocesador = build_preprocessor(X_train)
X_train_denso = preprocesador.fit_transform(X_train)
X_test_denso = preprocesador.transform(X_test)
print("Matriz densa:", X_train_denso.shape, "=", round(X_train_denso.nbytes / 1024**2, 1), "MB en float64")

pca = PCA(n_components=30, random_state=42).fit(X_train_denso)
varianza = pca.explained_variance_ratio_
acumulado = np.cumsum(varianza)
for objetivo in (0.80, 0.90, 0.95):
    componentes = int(np.searchsorted(acumulado, objetivo) + 1)
    print(f"Para explicar {objetivo:.0%} de la varianza hacen falta {componentes} componentes")

figura, (eje1, eje2) = plt.subplots(1, 2, figsize=(12.5, 4.8))
eje1.bar(range(1, len(varianza) + 1), varianza * 100, color="#0f7c7b")
eje1.plot(range(1, len(acumulado) + 1), acumulado * 100, color="#b45309", marker="o", markersize=3,
          label="varianza acumulada")
eje1.axhline(90, color="#475569", linestyle=":", linewidth=1)
eje1.set_xlabel("Componente principal")
eje1.set_ylabel("% de varianza explicada")
eje1.set_title("Espectro de PCA (ajustado solo con entrenamiento)")
eje1.legend(fontsize=8)

pca2 = PCA(n_components=2, random_state=42).fit(X_train_denso)
proyeccion = pca2.transform(X_test_denso)
for etiqueta, color in [("<=50K", "#64748b"), (">50K", "#b45309")]:
    mascara = (y_test == etiqueta).to_numpy()
    eje2.scatter(proyeccion[mascara, 0], proyeccion[mascara, 1], s=8, alpha=0.45, label=etiqueta, color=color)
eje2.set_xlabel("Componente 1")
eje2.set_ylabel("Componente 2")
eje2.set_title("Proyección 2D del conjunto de prueba")
eje2.legend(fontsize=8)
figura.tight_layout()
figura.savefig(REPORTS / "pca_analysis.png", dpi=160, bbox_inches="tight")
plt.close(figura)
print("Guardado:", REPORTS / "pca_analysis.png")

Matriz densa: (39073, 56) = 16.7 MB en float64
Para explicar 80% de la varianza hacen falta 11 componentes
Para explicar 90% de la varianza hacen falta 19 componentes
Para explicar 95% de la varianza hacen falta 26 componentes


Guardado: /home/trek/source/uasd/cienciadedatos/tareas/Unidad_1/INF8239_U01/reports/pca_analysis.png


## 5. Dos mapas t-SNE con semillas 42 y 7

`t-SNE` es costoso (depende del número de puntos), por lo que se aplica a una **submuestra estratificada
de 6.000 casos de prueba**; el submuestreo usa semilla fija, de modo que es reproducible. Se calculan
dos mapas con semillas distintas porque la posición de los puntos depende de la semilla: lo que se
compara es qué estructura **permanece** y qué cambia, no la coordenada de cada punto.

Una separación visual **no** demuestra causalidad ni que el modelo use esa estructura: t-SNE preserva
vecindades locales, no distancias globales.

In [7]:
rng = np.random.default_rng(42)
indice = rng.choice(len(X_test_denso), 6000, replace=False)
submuestra = X_test_denso[indice]
etiquetas = y_test.to_numpy()[indice]
print("Submuestra t-SNE:", submuestra.shape)

figura, ejes = plt.subplots(1, 2, figsize=(12.5, 5.4), sharex=False, sharey=False)
for eje, semilla in zip(ejes, [42, 7]):
    incrustacion = TSNE(n_components=2, random_state=semilla, init="pca", perplexity=30,
                        max_iter=1000, learning_rate="auto").fit_transform(submuestra)
    for etiqueta, color in [("<=50K", "#64748b"), (">50K", "#b45309")]:
        mascara = etiquetas == etiqueta
        eje.scatter(incrustacion[mascara, 0], incrustacion[mascara, 1], s=8, alpha=0.45,
                    label=etiqueta, color=color)
    eje.set_title(f"t-SNE · semilla {semilla}")
    eje.set_xticks([])
    eje.set_yticks([])
figura.suptitle("Adult · mapa t-SNE de 6.000 casos de prueba (submuestra estratificada, semilla 42)")
manijas, etiquetas = ejes[0].get_legend_handles_labels()
figura.legend(manijas, etiquetas, loc="lower right", fontsize=9, markerscale=2)
figura.tight_layout()
figura.savefig(REPORTS / "tsne_two_seeds.png", dpi=160, bbox_inches="tight")
plt.close(figura)
print("Guardado:", REPORTS / "tsne_two_seeds.png")

Submuestra t-SNE: (6000, 56)


Guardado: /home/trek/source/uasd/cienciadedatos/tareas/Unidad_1/INF8239_U01/reports/tsne_two_seeds.png


## 6. Decisión cuantificada

Se comparan la mejor configuración por F1 macro, la mejor configuración por costo y la elegida en la
frontera de Pareto, y se cuantifica la diferencia.

In [8]:
mejor_utilidad = resultados.loc[resultados["f1_macro"].idxmax()]
mejor_costo = resultados.loc[resultados["fit_time_median_s"].idxmin()]
elegida = resultados[resultados["is_pareto"]].sort_values("f1_macro", ascending=False).iloc[0]

print("Mejor F1 macro      :", mejor_utilidad["configuracion"], f"({mejor_utilidad['f1_macro']:.4f}, "
      f"{mejor_utilidad['fit_time_median_s']:.1f}s, {mejor_utilidad['model_size_kb']:.0f} KB)")
print("Menor tiempo de ajuste:", mejor_costo["configuracion"], f"({mejor_costo['f1_macro']:.4f}, "
      f"{mejor_costo['fit_time_median_s']:.1f}s, {mejor_costo['model_size_kb']:.0f} KB)")
print("Elegida (Pareto)    :", elegida["configuracion"], f"({elegida['f1_macro']:.4f}, "
      f"{elegida['fit_time_median_s']:.1f}s, {elegida['model_size_kb']:.0f} KB)")

# Alternativa dentro de la frontera y opción más económica del catálogo
frontera = resultados[resultados["is_pareto"]].sort_values("f1_macro", ascending=False)
alternativa = frontera.iloc[1] if len(frontera) > 1 else frontera.iloc[0]
mas_economica = resultados.loc[resultados["model_size_kb"].idxmin()]

decision = {
    "configuracion_elegida": elegida["configuracion"],
    "f1_macro_elegida": float(elegida["f1_macro"]),
    "f1_macro_maxima": float(mejor_utilidad["f1_macro"]),
    "diferencia_absoluta_f1": float(mejor_utilidad["f1_macro"] - elegida["f1_macro"]),
    "tiempo_elegida_s": float(elegida["fit_time_median_s"]),
    "tiempo_maxima_utilidad_s": float(mejor_utilidad["fit_time_median_s"]),
    "ahorro_tiempo_porcentaje": float((1 - elegida["fit_time_median_s"] / mejor_utilidad["fit_time_median_s"]) * 100),
    "tamano_elegida_kb": float(elegida["model_size_kb"]),
    "tamano_maxima_utilidad_kb": float(mejor_utilidad["model_size_kb"]),
    "diferencia_tamano_kb": float(mejor_utilidad["model_size_kb"] - elegida["model_size_kb"]),
    "inferencia_elegida_ms": float(elegida["predict_time_median_ms"]),
    "alternativa_en_la_frontera": alternativa["configuracion"],
    "f1_macro_alternativa": float(alternativa["f1_macro"]),
    "diferencia_f1_vs_alternativa": float(elegida["f1_macro"] - alternativa["f1_macro"]),
    "ahorro_tiempo_vs_alternativa_porcentaje": float(
        (1 - elegida["fit_time_median_s"] / alternativa["fit_time_median_s"]) * 100),
    "diferencia_tamano_vs_alternativa_kb": float(elegida["model_size_kb"] - alternativa["model_size_kb"]),
    "configuracion_mas_pequena": mas_economica["configuracion"],
    "f1_macro_mas_pequena": float(mas_economica["f1_macro"]),
    "diferencia_f1_vs_mas_pequena": float(elegida["f1_macro"] - mas_economica["f1_macro"]),
    "tamano_mas_pequena_kb": float(mas_economica["model_size_kb"]),
    "inferencia_mas_pequena_ms": float(mas_economica["predict_time_median_ms"]),
    "inferencia_svm_c10_ms": float(resultados.loc[resultados["configuracion"].eq("svm_c10"), "predict_time_median_ms"].iloc[0]),
    "ratio_inferencia_svm_sobre_boost": float(
        resultados.loc[resultados["configuracion"].eq("svm_c10"), "predict_time_median_ms"].iloc[0]
        / elegida["predict_time_median_ms"]),
    "tamano_rf_300_kb": float(resultados.loc[resultados["configuracion"].eq("rf_300"), "model_size_kb"].iloc[0]),
    "entorno": entorno,
    "seed": 42,
    "dataset": spec["nombre"],
    "licencia": spec["licencia"],
}
(REPORTS / "green_ai_decision.json").write_text(json.dumps(decision, indent=2), encoding="utf-8")
print(json.dumps(decision, indent=2))

Mejor F1 macro      : boost (0.8185, 0.6s, 360 KB)
Menor tiempo de ajuste: rf_100 (0.7823, 0.4s, 98807 KB)
Elegida (Pareto)    : boost (0.8185, 0.6s, 360 KB)
{
  "configuracion_elegida": "boost",
  "f1_macro_elegida": 0.8185273488230445,
  "f1_macro_maxima": 0.8185273488230445,
  "diferencia_absoluta_f1": 0.0,
  "tiempo_elegida_s": 0.5558288070023991,
  "tiempo_maxima_utilidad_s": 0.5558288070023991,
  "ahorro_tiempo_porcentaje": 0.0,
  "tamano_elegida_kb": 360.3642578125,
  "tamano_maxima_utilidad_kb": 360.3642578125,
  "diferencia_tamano_kb": 0.0,
  "inferencia_elegida_ms": 24.404810996202286,
  "alternativa_en_la_frontera": "rf_100",
  "f1_macro_alternativa": 0.7822652812143923,
  "diferencia_f1_vs_alternativa": 0.03626206760865214,
  "ahorro_tiempo_vs_alternativa_porcentaje": -46.619090982347245,
  "diferencia_tamano_vs_alternativa_kb": -98446.74609375,
  "configuracion_mas_pequena": "logreg",
  "f1_macro_mas_pequena": 0.7791674930665687,
  "diferencia_f1_vs_mas_pequena": 0.0393598

## 7. Conclusión

**La comparación homogénea de las ocho configuraciones se sostiene sobre el mismo dataset, el mismo
target y la misma partición, y la decisión se apoya en números, no en preferencias.** El F1 macro
va de 0,7749 (`logreg_pca`) a **0,8185 (`boost`)**: una dispersión de 0,0436, mucho menor que la del
Ejercicio 01, porque el problema no está en el modelo sino en que la variable que distingue los
ingresos altos (`capital_gain`) está ausente en el 91,74 % de las filas. El boosting (`HistGradientBoosting`)
gana con 0,8185 de F1 macro, 0,56 s de ajuste (mediana de tres repeticiones), 360 KB de pipeline
serializado y 24,4 ms de inferencia sobre 9.769 casos: **es a la vez la mejor utilidad y una de las
más baratas**, de modo que la diferencia absoluta de F1 macro frente a la mejor configuración es 0,00
y no hay penalidad que pagar. Frente a la otra configuración de la frontera, `rf_100` (0,7823), gana
0,0363 de F1 macro a cambio de 0,18 s más de ajuste y 98,4 MB menos de modelo.

**Sobre el costo real, que es donde el catálogo se separa:** el SVM con kernel RBF infiere **214 veces
más lento** que el boosting (5.228 s frente a 24,4 ms para las mismas 9.769 filas), y el Random Forest
de 300 árboles ocupa 296,9 MB serializados para un F1 macro de 0,7834, es decir, **824 veces más
modelo que el boosting con peor desempeño**. El PCA sí reduce el costo (SVM con PCA: 31,3 s y 2,3 MB
frente a 43,9 s y 6,0 MB sin PCA) pero **resta** 0,0038 de F1 macro: comprime la matriz densa de
58 columnas a 20 componentes y ese 90 % de varianza no incluye la señal que separa los ingresos altos.
Los dos mapas t-SNE (semillas 42 y 7) conservan los mismos agrupamientos —los casos `>50K` se concentran
en las mismas regiones— aunque cambian las coordenadas y la forma de los grupos: la estructura persiste,
la representación no, y una separación visual no demuestra causalidad ni que el modelo use esa geometría.

**Decisión: se entrega `boost`** (F1 macro 0,8185, 0,56 s, 360 KB, 24,4 ms), con la advertencia de que
el tiempo de ajuste y el tamaño del modelo son **proxies contextuales de costo, no consumo
energético**: las cifras solo valen en el hardware registrado en `reports/green_ai_decision.json`
(Python 3.12.15, scikit-learn 1.9.1, x86_64, Linux). El Ejercicio 02 no resuelve el problema de datos
del Ejercicio 01: el recall de `>50K` sigue en 0,66 y ninguna frontera de Pareto lo arregla.

In [9]:
print("Artefactos generados:")
for ruta in sorted(REPORTS.glob("*")):
    if ruta.suffix in {".csv", ".json", ".png"} and "green" in ruta.name or ruta.name in {
            "pareto.png", "pca_analysis.png", "tsne_two_seeds.png"}:
        print(f"  {ruta.name:28s} {ruta.stat().st_size / 1024:8.1f} KB")
print("Modelos serializados:", len(list(MODELS.glob('*.joblib'))))

Artefactos generados:
  green_ai_decision.json            1.4 KB
  green_ai_results.csv              1.7 KB
  pareto.png                      150.5 KB
  pca_analysis.png                278.9 KB
  tsne_two_seeds.png              374.0 KB
Modelos serializados: 8
